# 🐖 03 | Entrenamiento YOLO11n-seg

**RECURSO REQUERIDO: GPU (T4 o superior; selección antes de iniciar)**. Este notebook realiza exclusivamente su propia etapa. **No cambies CPU/GPU dentro del notebook.**

Proyecto: Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖. Datos: FaroPigSeg, licencia reportada CC BY-NC 4.0; citar fuente https://github.com/yudong888/FaroPig-Datasets.

**Ruta general:** `MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖/`. Ejecutar en orden 00 → 01 → 02 → 03 → 04. Todos los resultados se guardan en Drive.

## Objetivo
Entrenar **solo** YOLO11n-seg utilizando las listas de fechas creadas en Notebook 02. Conserva `best.pt`, `last.pt`, curvas y configuración en Google Drive. No realiza auditoría, descarga, evaluación sobre test ni segmentación de imágenes nuevas.

En Colab: **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU**, y entonces **Ejecutar todas**. No es necesario regresar a CPU en este notebook.

In [ ]:
import torch
assert torch.cuda.is_available(), ('NO HAY GPU: activa GPU en Entorno de ejecución → Cambiar tipo de entorno. '
                                  'No ejecutes el entrenamiento con CPU.')
print('✅ GPU disponible:',torch.cuda.get_device_name(0))

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "pyyaml>=6,<7" "pandas>=2,<3" openpyxl

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, os

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
if not ROOT.is_dir():
    raise FileNotFoundError(
        'No se encontró la carpeta principal. Verifica la cuenta de Google y el nombre de la carpeta: '
        'Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖'
    )
DATA_ROOT = ROOT / '01_Datasets' / 'FaroPigSeg' / 'extraido' / 'FaroPigSeg'
DATASET_DIR = ROOT / '01_Datasets' / 'FaroPigSeg'
RES = ROOT / '03_Resultados'
EXPO = ROOT / '05_Exportaciones_ZIP'
for sub in ('01_Exploracion','02_Segmentacion','03_Metricas','04_Figuras','05_Reportes','06_Entrenamiento'):
    (RES / sub).mkdir(parents=True, exist_ok=True)
EXPO.mkdir(parents=True,exist_ok=True)
print('Proyecto:',ROOT)

In [ ]:
import torch,yaml,subprocess,hashlib,platform,pandas as pd
from ultralytics import YOLO
assert torch.cuda.is_available(), 'Se perdió la GPU. Reinicia el entorno con GPU.'
YAML=DATASET_DIR/'split_fecha_v1'/'data_fecha.yaml'
assert YAML.is_file(), 'Primero ejecuta el notebook 02 (CPU) para crear data_fecha.yaml.'
cf=yaml.safe_load(YAML.read_text(encoding='utf8'))
for k in ('train','val','test'):
    assert Path(cf[k]).is_file(),f'Falta {cf[k]}. Ejecuta notebook 02.'
folder=RES/'06_Entrenamiento';name='YOLO11n_fecha_v1'
RUN=folder/name
BEST=RUN/'weights'/'best.pt'; LAST=RUN/'weights'/'last.pt'
print('Modelo: YOLO11n-seg | Épocas máximas: 60 | Tamaño 640 | Batch 8')
print('Corrida:',RUN)
print('Partición:',YAML)

if BEST.is_file() and (RUN/'ENTRENAMIENTO_FINALIZADO.json').exists():
    print('✅ Entrenamiento ya finalizado. No se repetirá.')
elif LAST.is_file():
    print('♻️ Reanudando entrenamiento desde last.pt de esta misma corrida.')
    YOLO(str(LAST)).train(resume=True)
else:
    model=YOLO('yolo11n-seg.pt')
    model.train(data=str(YAML),epochs=60,imgsz=640,batch=8,device=0,seed=2026,
                deterministic=True,patience=15,workers=2,project=str(folder),
                name=name,exist_ok=True,save=True,save_period=10,plots=True)

assert BEST.is_file(),'No existe best.pt tras el entrenamiento. Revisar la salida de Ultralytics.'
(RUN/'ENTRENAMIENTO_FINALIZADO.json').write_text(json.dumps({'completado_utc':datetime.now(timezone.utc).isoformat(),'yaml':str(YAML),'semilla':2026},ensure_ascii=False,indent=2),encoding='utf8')
print('✅ Pesos entrenados:',BEST)

In [ ]:
import ultralytics
history=RUN/'results.csv'
if history.is_file():
    df=pd.read_csv(history);df.columns=[x.strip() for x in df.columns]
    with pd.ExcelWriter(RES/'03_Metricas'/'CERDOS_03_Historial_Entrenamiento.xlsx',engine='openpyxl') as wr:
        df.to_excel(wr,sheet_name='Epocas',index=False)
    print('✅ Historial en Excel')
report={'etapa':'03_entrenamiento_GPU','fecha_utc':datetime.now(timezone.utc).isoformat(),
        'modelo':'yolo11n-seg.pt','semilla':2026,'epocas_max':60,'imgsz':640,'batch':8,
        'best_weights':str(BEST),'device':torch.cuda.get_device_name(0),
        'versions':{'torch':torch.__version__,'ultralytics':ultralytics.__version__,'python':platform.python_version()},
        'yaml':str(YAML),'advertencia':'El test no se evalúa aquí.'}
(RES/'05_Reportes'/'CERDOS_03_Entrenamiento_manifest.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf8')
print('✅ Entrenamiento registrado. Siguiente: notebook 04 en GPU.')

**Siguiente:** `04_CERDOS_Segmentacion_Evaluacion_GPU.ipynb`. Mantén la **GPU**, pero abre el notebook 04 por separado.